This notebook compares putative neural stem cell subclusters.

Input:  .h5ad with fine-grained neural stem cell annotations in ../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad  
Output: figures

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
sc.settings.set_figure_params(dpi=300)
import matplotlib.pyplot as plt
plt.rcParams['axes.grid'] = False

In [ ]:
adata = sc.read_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad')
adata

In [ ]:
nsc_adata = adata[adata.obs['cell_type'].isin(['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05'])].copy()
nsc_adata

In [ ]:
import scanpy.external as sce

# Harmony batch correction (creates X_pca_harmony)
sc.tl.pca(nsc_adata)
sce.pp.harmony_integrate(nsc_adata, key="patient_id", max_iter_harmony=20)

# Neighbors / UMAP / clustering on Harmony PCs
sc.pp.neighbors(nsc_adata, use_rep="X_pca_harmony")
sc.tl.umap(nsc_adata, min_dist=0.85)

In [ ]:
nsc_adata.uns['dumitru_nsc_colors'] = ['#00DCFF', 'purple']
sc.pl.umap(nsc_adata, color=['dumitru_nsc'])

In [ ]:
sc.pl.umap(nsc_adata, color=['donor'])
sc.pl.umap(nsc_adata, color=['age'], cmap='coolwarm', vmax=40)

In [ ]:
sc.pl.umap(nsc_adata, color=['total_counts'], cmap='coolwarm')

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import scanpy as sc

sc.pl.violin(
    nsc_adata, 
    keys='total_counts', 
    groupby='cell_type',
    palette=['darkblue', 'blue', 'green', 'orange', 'red'],
    inner='box',
    stripplot=False,
    rotation=90
)

# Pairwise significance testing
groups = ['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05']
pairs = list(itertools.combinations(groups, 2))
pvals = []

for g1, g2 in pairs:
    # Get values for each group
    v1 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g1, 'total_counts']
    v2 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g2, 'total_counts']
    
    # Run Mann-Whitney
    _, p = stats.mannwhitneyu(v1, v2)
    pvals.append(p)

# Apply Holm correction
_, adj_p, _, _ = multipletests(pvals, method='holm')

# Print results
for (g1, g2), p_adj in zip(pairs, adj_p):
    if p_adj < 0.05:
        print(f"{g1} vs {g2}: p_adj = {p_adj:.2e}")

In [ ]:
sc.pl.umap(nsc_adata, color=['n_genes_by_counts'], cmap='coolwarm')

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import scanpy as sc

sc.pl.violin(
    nsc_adata, 
    keys='n_genes_by_counts', 
    groupby='cell_type',
    palette=['darkblue', 'blue', 'green', 'orange', 'red'],
    inner='box',
    stripplot=False,
    rotation=90,
    show=False
)

plt.tight_layout()
plt.savefig("../Figures/unique_genes_violin.svg", format="svg", bbox_inches="tight")
plt.show()

# Pairwise significance testing
groups = ['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05']
pairs = list(itertools.combinations(groups, 2))
pvals = []

for g1, g2 in pairs:
    # Get values for each group
    v1 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g1, 'n_genes_by_counts']
    v2 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g2, 'n_genes_by_counts']
    
    # Run Mann-Whitney
    _, p = stats.mannwhitneyu(v1, v2)
    pvals.append(p)

# Apply Holm correction
_, adj_p, _, _ = multipletests(pvals, method='holm')

# Print results
for (g1, g2), p_adj in zip(pairs, adj_p):
    if p_adj < 0.05:
        print(f"{g1} vs {g2}: p_adj = {p_adj:.2e}")

# Cell cycle scoring

In [ ]:
cell_cycle_genes = [x.strip() for x in open('../Data/regev_lab_cell_cycle_genes.txt')]
print(len(cell_cycle_genes))

# Split into 2 lists
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

cell_cycle_genes = [x for x in cell_cycle_genes if x in nsc_adata.var_names]
print(len(cell_cycle_genes))

In [ ]:
sc.tl.score_genes_cell_cycle(nsc_adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
sc.pl.umap(nsc_adata, color=['phase', 'S_score', 'G2M_score'], cmap='coolwarm')

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import scanpy as sc

sc.pl.violin(
    nsc_adata, 
    keys='S_score', 
    groupby='cell_type',
    palette=['darkblue', 'blue', 'green', 'orange', 'red'],
    inner='box',
    stripplot=False,
    rotation=90
)

# Pairwise significance testing
groups = ['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05']
pairs = list(itertools.combinations(groups, 2))
pvals = []

for g1, g2 in pairs:
    # Get values for each group
    v1 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g1, 'S_score']
    v2 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g2, 'S_score']
    
    # Run Mann-Whitney
    _, p = stats.mannwhitneyu(v1, v2)
    pvals.append(p)

# Apply Holm correction
_, adj_p, _, _ = multipletests(pvals, method='holm')

# Print results
for (g1, g2), p_adj in zip(pairs, adj_p):
    if p_adj < 0.05:
        print(f"{g1} vs {g2}: p_adj = {p_adj:.2e}")

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import scanpy as sc

sc.pl.violin(
    nsc_adata, 
    keys='G2M_score', 
    groupby='cell_type',
    palette=['darkblue', 'blue', 'green', 'orange', 'red'],
    inner='box',
    stripplot=False,
    rotation=90,
    show=False
)

plt.tight_layout()
plt.savefig("../Figures/g2m_score_violin.svg", format="svg", bbox_inches="tight")
plt.show()

# Pairwise significance testing
groups = ['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05']
pairs = list(itertools.combinations(groups, 2))
pvals = []

for g1, g2 in pairs:
    # Get values for each group
    v1 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g1, 'G2M_score']
    v2 = nsc_adata.obs.loc[nsc_adata.obs['cell_type'] == g2, 'G2M_score']
    
    # Run Mann-Whitney
    _, p = stats.mannwhitneyu(v1, v2)
    pvals.append(p)

# Apply Holm correction
_, adj_p, _, _ = multipletests(pvals, method='holm')

# Print results
for (g1, g2), p_adj in zip(pairs, adj_p):
    if p_adj < 0.05:
        print(f"{g1} vs {g2}: p_adj = {p_adj:.2e}")

# Pseudotime

In [ ]:
sc.pl.umap(nsc_adata, color=['cell_type'], legend_loc='on data')

In [ ]:
# Compute diffusion map
sc.tl.diffmap(nsc_adata)

# Set root cell for pseudotime
root_idx = nsc_adata.obs['cell_type'][nsc_adata.obs['cell_type'] == 'nsc_01'].index[38]
nsc_adata.uns['iroot'] = np.where(nsc_adata.obs_names == root_idx)[0][0]

# Compute DPT pseudotime
sc.tl.dpt(nsc_adata) 
# Visualize pseudotime
sc.pl.umap(nsc_adata, color=['dpt_pseudotime'], title='Pseudotime', size=200, frameon=False)

In [ ]:
import matplotlib.pyplot as plt

# Define genes to track along pseudotime
gene_names = ['ETNPPL', 'GLI3', 'CNTN1', 'PAMR1', 'TRPM3', 'RYR3',
         'RHOJ', 'HOPX', 'SLC4A4', 'ID4', 'LRRC3B', 'GRM3', 'GPC5', 'SLC1A2', 'GLUL',
         'ALDOC', 'FABP7', 'TNC', 'VIM', 'VCAN',
         'SOX2', 'PAX6', 'PROX1', 'SYNE2', 'NES', 'ASCL1', 'EGFR', 'PTPRD', 'MYO16',
         'EZH2', 'STMN1', 'MKI67']

# Create subset with genes of interest
d = nsc_adata[:, gene_names].copy()

# Use raw counts and normalize
d.X = d.layers['counts'].toarray()
for i, gene in enumerate(gene_names):
    gene_counts = d.X[:, i]
    max_val = np.percentile(gene_counts, 75)
    d.X[:, i] = np.clip(gene_counts / max_val, 0, 1)

# Use existing pseudotime
d.obs['pseudotime'] = nsc_adata.obs['dpt_pseudotime']

# Plot genes along pseudotime trajectory
plt.style.use("default")
sc.set_figure_params(figsize=(5, 5))

sc.pl.paga_path(
    d, 
    ['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05'],
    gene_names,
    groups_key='cell_type',
    annotations=['pseudotime'],
    color_map='bwr',
    color_maps_annotations={'pseudotime': 'viridis'},
    n_avg=30,
    show_yticks=True,
    ytick_fontsize=10,
    title_fontsize=20,
    show_node_names=False
)